# Randomized Genome Validation

This notebook reproduces the randomized genome validation analyses presented in the accompanying study.

The notebook was developed and executed in the Kaggle Notebook environment. Input paths assume the corresponding datasets are available as Kaggle datasets.

# Imports

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")
import re
import optuna
from optuna.samplers import TPESampler
import pickle
import matplotlib.pyplot as plt
import scipy.stats as stats

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import shap

from sklearn.model_selection import train_test_split
import os

# Data Preparation

In [ ]:
np.set_printoptions(threshold=np.inf)

# Load dataset
features_matrix = pd.read_csv('../input/wt25-dataset/Xtbl_wt25.csv')
X = features_matrix.drop(columns=['transcript', 'random_factor', 'random_num'])

df = pd.read_csv('../input/wt25-dataset/Ytbl_wt25.csv')
y = df['RE'].copy()

categorical_cols = [col for col in X.columns if X[col].dtype == 'object' or X[col].dtype.name == 'category']

# Setup OneHotEncoder
encoder = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', encoder, categorical_cols),
    ],
    remainder='passthrough' 
)

# Fit the preprocessor on the entire dataset
preprocessor.fit(X)

# Data Preparation Helper

In [ ]:
# Data Preparation for randomized genome tables

def prepare_randgenome_table(tbl_idx):
    """
    Load Xtbl_rand_<tbl_idx>.csv, drop non-feature columns,
    and transform with the already-fitted preprocessor.

    Returns:
        raw_df:  original dataframe (for IDs like 'transcript')
        X_prepared: transformed feature matrix ready for model.predict()
    """
    path = f"/kaggle/input/randgenome-x/Xtbl_rand_{tbl_idx}.csv"
    raw_df = pd.read_csv(path)

    # Use the same feature selection as for the training data:
    X_new = raw_df.drop(columns=['transcript', 'random_factor', 'random_num'])

    X_prepared = preprocessor.transform(X_new)

    return raw_df, X_prepared


# Tuning

In [ ]:
# XGB

def return_rmse(dtrain, dtest, params):
    model = xgb.train(params, dtrain, num_boost_round=600, evals=[(dtest, 'eval')], early_stopping_rounds=100,
                  verbose_eval=0)
    result = model.eval(dtest)
    result = float(re.search(r'[\d.]+$', result).group(0))
    return result

def objective(dtrain, dtest, trial):
    params = {'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 10.0),
    'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 10.0),
    'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
    'gamma': trial.suggest_float('gamma', 0, 5),
    'subsample': trial.suggest_float('subsample', 0.5, 1.0),
    'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
    'max_depth': trial.suggest_int('max_depth', 3, 10),
    'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    'min_split_loss': trial.suggest_float('min_split_loss', 0, 5),
    'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
             'nthread': -1}
    return return_rmse(dtrain, dtest, params)

def hyperparam_tuning(X_train, y_train, X_test, y_test):
    res=[]
    dtrain = xgb.DMatrix(X_train, y_train, enable_categorical=True)
    dtest = xgb.DMatrix(X_test, y_test, enable_categorical=True)
    res.append(return_rmse(dtrain, dtest, params={}))

    xgb_study = optuna.create_study(direction='minimize',sampler =TPESampler() )
    xgb_study.optimize(lambda trial: objective(dtrain, dtest, trial), n_trials=100, show_progress_bar=True)
    
    return xgb_study


# lr

def ridge_objective(X_train, y_train, X_test, y_test, trial):
    alpha = trial.suggest_float("alpha", 0.01, 100.0, log=True)

    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    return rmse

def tune_ridge_with_optuna(X_train, y_train, X_test, y_test):
    lr_study = optuna.create_study(direction="minimize", sampler=TPESampler())
    lr_study.optimize(lambda trial: ridge_objective(X_train, y_train, X_test, y_test, trial), 
                   n_trials=50, 
                   show_progress_bar=True)
    
    return lr_study


# lasso

def lasso_objective(X_train, y_train, X_test, y_test, trial):
    alpha = trial.suggest_float("alpha", 0.0001, 1.0, log=True)

    model = Lasso(alpha=alpha, random_state=42, max_iter=10000)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    return rmse

def tune_lasso_with_optuna(X_train, y_train, X_test, y_test):
    lasso_study = optuna.create_study(direction="minimize", sampler=TPESampler())
    lasso_study.optimize(lambda trial: lasso_objective(X_train, y_train, X_test, y_test, trial), 
                   n_trials=50, 
                   show_progress_bar=True)
    
    return lasso_study


# rf

def rf_objective(X_train, y_train, X_test, y_test, trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'random_state': 42,
        'n_jobs': -1
    }

    model = RandomForestRegressor(**params)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    return rmse

def tune_rf_with_optuna(X_train, y_train, X_test, y_test):
    rf_study = optuna.create_study(direction="minimize", sampler=TPESampler())
    rf_study.optimize(lambda trial: rf_objective(X_train, y_train, X_test, y_test, trial), 
                   n_trials=50, 
                   show_progress_bar=True)
    
    return rf_study

# Train Models

In [ ]:
def train_models(X_train, y_train, xgb_study, best_lasso_alpha, rf_params):

    # lr
    lr_pipeline = Pipeline([
        ('regressor',  LinearRegression())
    ])
    lr_pipeline.fit(X_train, y_train)

    # lasso
    lasso_pipeline = Pipeline([
        ('regressor', Lasso(alpha=best_lasso_alpha, random_state=42, max_iter=10000))
    ])
    lasso_pipeline.fit(X_train, y_train)

    # rf
    rf_pipeline = Pipeline([
        ('regressor', RandomForestRegressor(**rf_params))
    ])
    rf_pipeline.fit(X_train, y_train)
    
    # XGBoost
    xgb_model = xgb.XGBRegressor(objective='reg:squarederror', **xgb_study.best_params)  # Assuming best_params are already set
    xgb_model.fit(X_train, y_train)
    
    return {
        'lr': {'model': lr_pipeline},
        'lasso': {'model': lasso_pipeline},
        'rf': {'model': rf_pipeline},
        'xgb': {'model': xgb_model}
    }

# Run Experiment

In [ ]:
# Initialize dictionary to store feature importances for each model across all splits
feature_importances = {
    'lr': [],
    'lasso': [],
    'rf': [],
    'xgb': []
}

# Transform the data using the pre-fitted preprocessor
X_all = preprocessor.transform(X)
y_all = y.values

# 80% for training, 20% for validation
X_train, X_val, y_train, y_val = train_test_split(
    X_all,
    y_all,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("Train shape:", X_train.shape, "Validation shape:", X_val.shape)

xgb_study = hyperparam_tuning(X_train, y_train, X_val, y_val)
lasso_study = tune_lasso_with_optuna(X_train, y_train, X_val, y_val)
best_lasso_alpha = lasso_study.best_params['alpha']
rf_study = tune_rf_with_optuna(X_train, y_train, X_val, y_val)
rf_params = rf_study.best_params

model_dict = train_models(X_all, y_all, xgb_study, best_lasso_alpha, rf_params)

# Predict labels for 20 randomized genome tables (Xtbl_rand_1..20)
# using ONLY the trained XGB model

output_dir = "/kaggle/working/randgenome_predictions"
os.makedirs(output_dir, exist_ok=True)

# Get the trained XGB model
xgb_model = model_dict['xgb']['model']

for tbl_idx in range(1, 21):
    print(f"\nProcessing Xtbl_rand_{tbl_idx}.csv ...")

    raw_df, X_rand = prepare_randgenome_table(tbl_idx)

    # Predict labels
    y_pred = xgb_model.predict(X_rand)

    # Build output dataframe
    pred_df = pd.DataFrame({
        "transcript": raw_df["transcript"],
        "RE_pred_xgb": y_pred
    })

    out_path = os.path.join(output_dir, f"Ypred_rand_{tbl_idx}.csv")
    pred_df.to_csv(out_path, index=False)

    print(f"Saved predictions for table {tbl_idx} -> {out_path}")


# OOF Predictions

In [ ]:
from sklearn.model_selection import KFold

# 5-fold OOF predictions with XGBoost on WT25

# Transform full WT25 feature matrix
X_all = preprocessor.transform(X)
y_all = y.values
n_samples = X_all.shape[0]

# Array to hold OOF predictions
oof_pred = np.zeros(n_samples, dtype=float)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_rmse = []
fold_r2 = []

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_all), start=1):
    print(f"\n=== Fold {fold_idx} ===")
    
    X_tr, X_val = X_all[train_idx], X_all[val_idx]
    y_tr, y_val = y_all[train_idx], y_all[val_idx]
    
    # XGBoost model with tuned hyperparameters from xgb_study
    xgb_model_cv = xgb.XGBRegressor(
        objective='reg:squarederror',
        **xgb_study.best_params
    )
    xgb_model_cv.fit(X_tr, y_tr)
    
    # Predict on held-out fold
    y_val_pred = xgb_model_cv.predict(X_val)
    oof_pred[val_idx] = y_val_pred
    
    # Per-fold metrics
    rmse = mean_squared_error(y_val, y_val_pred, squared=False)
    r2 = r2_score(y_val, y_val_pred)
    fold_rmse.append(rmse)
    fold_r2.append(r2)

print("OOF Mean RMSE:", np.mean(fold_rmse))
print("OOF Mean R²  :", np.mean(fold_r2))

# Load original Y table to get gene names (and keep RE)
ytbl = pd.read_csv("../input/wt25-dataset/Ytbl_wt25.csv")  # columns: gene, RE

# Sanity check
print(ytbl.shape, len(oof_pred))

# Build a table with original and OOF predictions
oof_df = ytbl.copy()
oof_df = oof_df.rename(columns={"RE": "original_RE"})
oof_df["RE_pred_oof_xgb"] = oof_pred

display(oof_df.head())

oof_out_path = "/kaggle/working/OOF_predictions_xgb_5fold.csv"
oof_df.to_csv(oof_out_path, index=False)
print("Saved 5-fold OOF predictions to:", oof_out_path)


# Compare mean RE

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt

OOF_PATH = "/kaggle/working/OOF_predictions_xgb_5fold.csv"
PRED_DIR = "/kaggle/working/randgenome_predictions" 

# Load OOF predictions and compute mean
oof_df = pd.read_csv(OOF_PATH)
oof_mean = oof_df["RE_pred_oof_xgb"].mean()

print(f"Mean OOF predicted RE (WT25, 5-fold CV): {oof_mean:.4f}")

# Load all 20 randomized genome predictions
#    and compute mean per genome
rand_means = []

for i in range(1, 21):
    fname = f"Ypred_rand_{i}.csv"
    fpath = os.path.join(PRED_DIR, fname)
    df = pd.read_csv(fpath)

    mean_i = df["RE_pred_xgb"].mean()
    rand_means.append({
        "rand_idx": i,
        "mean_RE_pred_xgb": mean_i
    })

rand_means_df = pd.DataFrame(rand_means)
display(rand_means_df)

# Overall mean across randomized genomes
overall_rand_mean = rand_means_df["mean_RE_pred_xgb"].mean()

print(f"\nMean RE across all 20 randomized genomes: {overall_rand_mean:.4f}")
print(f"OOF mean vs overall random mean: {oof_mean:.4f} vs {overall_rand_mean:.4f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# per-genome mean predicted RE (randomized) with OOF baseline 

x = rand_means_df["rand_idx"].values
y = rand_means_df["mean_RE_pred_xgb"].values

n_above = (y > oof_mean).sum()
n_total = len(y)

plt.figure(figsize=(10, 5))

# dots: mean per randomized genome
plt.scatter(x, y, alpha=0.9)

# OOF baseline line
plt.axhline(oof_mean, linestyle="--", linewidth=2)

# Axis labels / title
plt.xlabel("Randomized genome index")
plt.ylabel("Mean predicted RE (XGB)")
plt.title("Mean predicted RE per randomized genome vs OOF baseline")

plt.xticks(np.arange(1, 21, 1))
plt.grid(True)
plt.tight_layout()
plt.show()

# RE Histograms

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

PRED_DIR = "/kaggle/working/randgenome_predictions"

# Measured RE
real_RE = wt["RE"].dropna().values

# OOF predicted RE
oof_RE = oof_df["RE_pred_oof_xgb"].dropna().values

# Randomized genomes: mean RE per gene
rand_mat = []

for i in range(1, 21):
    df = pd.read_csv(os.path.join(PRED_DIR, f"Ypred_rand_{i}.csv"))
    rand_mat.append(df["RE_pred_xgb"].values)

rand_mat = np.vstack(rand_mat)          # shape: (20, n_genes)
rand_RE_mean = rand_mat.mean(axis=0)    # mean over genomes, per gene

# Shared bins
all_vals = np.concatenate([real_RE, oof_RE, rand_RE_mean])
bins = np.linspace(all_vals.min(), all_vals.max(), 50)

# Compute means
mean_real = real_RE.mean()
mean_oof = oof_RE.mean()
mean_rand = rand_RE_mean.mean()

# Plot histograms + mean lines
plt.figure(figsize=(9, 6))

plt.hist(real_RE, bins=bins, density=True, alpha=0.5, label="Measured RE")
plt.hist(oof_RE, bins=bins, density=True, alpha=0.5, label="OOF predicted RE")
plt.hist(rand_RE_mean, bins=bins, density=True, alpha=0.5, label="Random-genome mean RE")

# Mean lines
plt.axvline(mean_real, linestyle="--", linewidth=2, label=f"Measured mean = {mean_real:.2f}")
plt.axvline(mean_oof, linestyle="--", linewidth=2, label=f"OOF mean = {mean_oof:.2f}")
plt.axvline(mean_rand, linestyle="--", linewidth=2, label=f"Random mean = {mean_rand:.2f}")

plt.xlabel("RE value")
plt.ylabel("Density")
plt.title("Distribution of RE values: measured vs OOF vs randomized (mean)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()